# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [1]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_397/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [7]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [8]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [9]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [10]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [11]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [12]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [13]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [14]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 91%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [15]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

---
# Відповіді на теоретичні питання 0.1 (питання 21–30)

### 21. Як довго виконувалися клітинки ноутбука 03_rag_langchain_eval? Чи були проблеми?

Google Colab, GPU T4.

| Клітинка | Час |
|---|---|
| Встановлення бібліотек (LangChain, FAISS, BM25) | 19 с |
| Імпорти | 46 с |
| Завантаження й очищення 14 PDF | 54 с |
| Розбиття на фрагменти (385) | < 1 с |
| Модель ембедингів + векторне сховище FAISS | 34 с |
| BM25, гібридний пошук | < 1 с |
| Завантаження cross-encoder `bge-reranker-v2-m3` (2.3 GB) | 23 с |
| Метрики ретриву для 4 методів на 23 питаннях | 8 с |
| **Завантаження генеративної моделі Qwen3-4B (8 GB)** | **≈ 4–5 хв** |
| Один запит RAG-ланцюжка | 3 с |
| **Оцінювання 26 питань (генерація + суддя)** | **≈ 1–2 хв** |

Разом ≈ 10 хвилин; найдовше — завантаження генеративної моделі.

**Проблеми:** помилок, які зупинили б виконання, не було. Попередження:
- при встановленні: `ERROR: pip's dependency resolver… google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2`. Це конфлікт версій з передвстановленою в Colab бібліотекою; на роботу ноутбука не вплинув;
- `DeprecationWarning: langchain-community is being sunset and is no longer actively maintained` — пакет `langchain-community` (звідки беруться FAISS і BM25) більше не розвивається, у майбутньому потрібно буде перейти на окремі пакети інтеграцій;
- `You seem to be using the pipelines sequentially on GPU` — рекомендація обробляти запити батчами для швидкості;
- `unauthenticated requests to the HF Hub`.

### 22. Чим RecursiveCharacterTextSplitter відрізняється від розбиття за реченнями з демо 2? Для чого keep_separator="end"?

**RecursiveCharacterTextSplitter** (`chunk_size=1000`, `chunk_overlap=200`, `separators=[". ", "; ", ", ", " ", ""]`):
- ріже текст **за першим роздільником зі списку** (кінець речення `. `). Якщо шматок усе ще довший за 1000 символів, ріже його **наступним** роздільником (`; `, потім `, `, потім пробіл, в крайньому разі будь-де);
- потім **склеює** дрібні шматки у фрагменти до 1000 символів;
- **перекриття** задається **в символах** (200), тобто сусідні фрагменти мають спільний «хвіст» із кількох шматків до 200 символів;
- результат: **385 фрагментів**.

**Розбиття за реченнями в демо 2:**
- ділить текст на речення **регулярним виразом**, який враховує особливості документів: номер пункту на зразок «1.4.» не вважається кінцем речення, а наступне слово має починатися з великої літери чи цифри;
- задовге «речення» (таблиця) ріжеться просто **за довжиною**, без проміжних роздільників;
- перекриття — **рівно одне останнє речення**;
- результат: 410 фрагментів.

Головні відмінності: Recursive має **кілька рівнів запасних роздільників** (тому довгі таблиці ріже акуратніше, по `; ` чи `, `), але **не розуміє нумерації пунктів**: `". "` у «1.4. Староста…» для нього межа, тож номер пункту може відірватися від тексту пункту. Розбиття з демо 2 «розумніше» щодо змісту документів, але написане вручну.

**`keep_separator="end"`** — роздільник (`. `, `; `…) **залишається в кінці попереднього шматка**, а не переноситься на початок наступного. Тоді речення закінчується своєю крапкою, а наступний фрагмент починається з великої літери, а не з «. Студент…». Фрагменти виглядають природно для читання й цитування.

### 23. Чому для FAISS обрано DistanceStrategy.MAX_INNER_PRODUCT? Чи можна так робити без нормалізації ембедингів?

За замовчуванням FAISS шукає за **евклідовою відстанню (L2)**: менше значення означає ближчий фрагмент. `MAX_INNER_PRODUCT` шукає за **скалярним добутком**: більше значення означає ближчий фрагмент.

Ембединги **нормалізовані** (`normalize_embeddings=True` і для документів, і для запитів), тобто довжина кожного вектора дорівнює 1. Для таких векторів **скалярний добуток дорівнює косинусній подібності** (питання 1). Тому `MAX_INNER_PRODUCT` фактично шукає за косинусом, саме тією мірою, якою ми користувалися в демо 1 і 2, і повертає звичні оцінки подібності (0.60, 0.53…), де більше означає краще.

(Для нормалізованих векторів L2 дав би той самий порядок результатів, бо ‖a − b‖² = 2 − 2·cos, але оцінки були б відстанями, а не подібностями.)

**Без нормалізації так робити не можна** (точніше, результат перестане бути косинусною подібністю). Скалярний добуток `a · b = ‖a‖ · ‖b‖ · cos` залежить і від **довжини** векторів: фрагменти з «довшими» ембедингами отримували б вищі оцінки незалежно від змісту, і пошук віддавав би їм перевагу. Тоді треба або нормалізувати вектори перед додаванням в індекс, або використовувати косинусну відстань.

### 25. Бал RRF для фрагмента на 2-й позиції у векторному пошуку й на 5-й у BM25. Чому RRF використовує лише позиції?

Формула RRF (reciprocal rank fusion): кожен список дає фрагменту `1 / (c + r)` балів, де `r` — позиція фрагмента в списку, `c = 60`. Бали з усіх списків додаються.

Для фрагмента на 2-й позиції у векторному пошуку і на 5-й у BM25:

`RRF = 1 / (60 + 2) + 1 / (60 + 5) = 1/62 + 1/65 ≈ 0.016129 + 0.015385 ≈ 0.031514`

(обчислення кодом — у клітинці нижче).

**Чому лише позиції, а не самі оцінки:**
- оцінки двох методів **у різних шкалах і незрівнянні**: косинусна подібність лежить приблизно в межах 0.3–0.8, а BM25 — необмежене число (5, 12, 30…), яке залежить від довжини запиту, частоти слів і корпусу. Просто додати їх не можна: одна з оцінок повністю переважила б іншу;
- щоб поєднувати оцінки, треба було б їх нормалізувати й підбирати ваги, а це додаткові налаштування, що залежать від корпусу. **Позиції ж мають однакову шкалу** для будь-якого методу: перше місце в будь-якому списку означає «найкращий кандидат цього методу»;
- RRF стійкий до «викидів»: одна аномально висока оцінка не перекреслює рішення іншого методу;
- константа `c = 60` згладжує різницю між сусідніми позиціями: фрагмент, що стоїть досить високо **в обох** списках, отримує більше балів, ніж фрагмент, перший лише в одному.

In [16]:
# Питання 25: бал RRF «вручну», без бібліотек
c = 60
vector_rank, bm25_rank = 2, 5
score = 1 / (c + vector_rank) + 1 / (c + bm25_rank)
print(f"1/(60+2) = {1 / (c + vector_rank):.6f}")
print(f"1/(60+5) = {1 / (c + bm25_rank):.6f}")
print(f"RRF = {score:.6f}")

# Для порівняння: фрагмент лише на 1-му місці в одному списку і відсутній в іншому
print(f"Лише 1-ше місце в одному списку: {1 / (c + 1):.6f}")

1/(60+2) = 0.016129
1/(60+5) = 0.015385
RRF = 0.031514
Лише 1-ше місце в одному списку: 0.016393


### 26. Чому cross-encoder застосовують лише до 20 кандидатів? Скільки проходів моделі знадобилося б для всього корпусу?

**Bi-encoder** (модель ембедингів) кодує запит і фрагменти **окремо**. Ембединги всіх 385 фрагментів рахуються **один раз заздалегідь**, а для кожного питання потрібен лише **1 прохід** моделі (запит) і швидке множення матриць.

**Cross-encoder** (`bge-reranker-v2-m3`, ≈ 568 млн параметрів) отримує **запит і фрагмент разом** і оцінює їхню відповідність. Тому заздалегідь нічого не порахуєш: **для кожної пари «питання — фрагмент» потрібен окремий прохід моделі**, і кожен прохід обробляє ~300–500 токенів.

- Переранжування 20 кандидатів: **20 проходів** на питання;
- переранжування всього корпусу: **385 проходів** на питання, тобто **≈ 19 разів більше** часу й обчислень. Для бенчмарку з 26 питань це ≈ 10 000 проходів замість 520.

У реальних системах із тисячами чи мільйонами фрагментів переранжування всього корпусу взагалі неможливе: одне питання оброблялося б хвилини чи години.

Тому використовують **двоетапну схему**: швидкий пошук (векторний + BM25) відбирає ~20 найімовірніших кандидатів, а повільна, але точніша модель переставляє лише їх. Майже всі правильні фрагменти вже є серед перших 20 (hit@5 для гібридного пошуку вже 0.91), тож обмеження до 20 кандидатів майже не шкодить якості.

### 28. Порівняння метрик ретриву для чотирьох методів. Чому переранжування найсильніше впливає на hit@1?

| Метод | hit@1 | hit@5 | MRR |
|---|---|---|---|
| vector_search (семантичний) | 0.61 | 0.87 | 0.74 |
| bm25_search (лексичний) | 0.52 | 0.65 | 0.59 |
| hybrid_search (RRF) | 0.61 | 0.91 | 0.76 |
| **rerank_search** (гібрид + cross-encoder) | **0.83** | **0.96** | **0.90** |

**Порівняння:**
- **BM25 найслабший** на українських текстах: слова мають багато форм («гуртожиток» / «гуртожитку» / «гуртожитками» для нього різні), а питання рідко дослівно повторюють формулювання документа;
- **семантичний пошук** помітно кращий: розуміє синоніми й різні форми слів;
- **гібридний** не змінив hit@1 (0.61), але підняв hit@5 з 0.87 до 0.91: BM25 «підтягує» фрагменти з рідкісними точними збігами (числа, коди, терміни), які семантичний пошук ставив нижче;
- **переранжування** дає найбільший приріст за всіма метриками: hit@1 0.61 → **0.83** (+22 пункти), MRR 0.76 → **0.90**.

**Чому переранжування найсильніше впливає саме на hit@1:**
1. Cross-encoder **не шукає нові фрагменти**, а лише **переставляє** 20 кандидатів, знайдених гібридним пошуком. Якщо правильного фрагмента немає серед кандидатів, переранжування не допоможе. Тому hit@5 (і тим більше hit@20) обмежений першим етапом і зростає мало (0.91 → 0.96).
2. Зате серед кандидатів правильний фрагмент **уже є майже завжди**, просто часто не на першому місці. Cross-encoder читає запит і фрагмент **разом**, бачить, чи справді фрагмент відповідає на питання, і переносить правильний фрагмент **нагору**. Саме це і вимірює hit@1.

**Застереження:** питань лише 23, одне питання ≈ 4.3 пункти, тож різниці в кілька пунктів можуть бути випадковими. Але приріст hit@1 на 22 пункти (≈ 5 питань) — помітний ефект.

### 30. Аналіз неправильних відповідей (розділ 10): тип помилки

Загальні результати: **91%** правильних відповідей на 23 питання з відповіддю (21 з 23), **96%** коректних цитувань (22 з 23), **100%** правильних відмов на 3 питання без відповіді. Неправильними визнано 2 відповіді:

**1. «Для кого встановлюється стипендія ректора?»** — **помилка судді / неоднозначний еталон.**
- Правильний документ 344 знайдено на 1-му місці, цитування коректне.
- Еталон: «Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.»
- Відповідь системи: «Для студентів денної форми навчання, які мають суттєві досягнення в навчанні, науковій, творчій та громадській діяльності, **є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань** [1].»
- Відповідь містить **усе з еталона**, тобто вона правильна, і **доповнює його деталями** з того самого документа. Суддя відповів «ні», мабуть, сприйнявши додаткові умови як «суперечність» або «іншу інформацію». Еталон коротший, ніж формулювання в документі, тому суддя не може перевірити зайві подробиці. **Виправлення:** уточнити промпт судді («додаткові деталі з документа не є помилкою») або доповнити еталон.

**2. «Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?»** — **помилка генерації через застарілі й суперечливі джерела** (з елементами промаху ретриву).
- Еталон: F2 (документ 3320). Знайдено: [3329, 3329, **3320**, 3329, 3329], тобто 4 з 5 фрагментів зі **застарілої** програми 2024 року, а чинна лише на 3-му місці.
- Відповідь: «… — **121 [2]**», з посиланням на фрагмент зі старої програми.
- Формально ретрив «знайшов» правильний документ, але переважно повернув застарілу редакцію з майже однаковою назвою. Модель обрала шифр зі старого документа, хоча в питанні є слово «чинній», і процитувала його. До того ж, як показано в демо 2, навіть чинна програма в кількох розділах містить старий шифр 121.
- **Виправлення:** фільтрувати застарілі документи за метаданими (`exclude`), додавати в метадані дату й статус «чинний», виправити суперечності в самому документі (див. питання 19).

**Висновок:** з двох «неправильних» відповідей одна насправді правильна (помилка оцінювання), тож реальна точність системи вища за 91%. Друга — системна проблема якості джерел, яку не виправити налаштуваннями пошуку чи промпту.

In [17]:
# Питання 24: на якій позиції правильний фрагмент у семантичному пошуку і в BM25
my_questions = [
    # питання з рідкісними точними словами, кодами, числами
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат"),
    ("Скільки кредитів ЄКТС має бакалаврська програма F2?", ["3320"], r"240 кредит"),
    ("Якому діапазону балів відповідає FX?", ["275"], r"35 ?- ?59"),
    # питання, сформульовані іншими словами, ніж у документі
    ("Чи можна поселити котика в кімнату гуртожитку?", ["1219", "348"], r"утримувати тварин"),
    ("Коли треба заплатити за житло на весняний семестр?", ["348"], r"1 лютого"),
    ("О котрій годині в гуртожитку потрібно не шуміти?", ["1219"], r"23:00"),
]
rows = []
for q, gold, evidence in my_questions:
    rows.append({
        "питання": q,
        "семантичний (позиція)": first_hit_rank(vector_search(q, CANDIDATES), gold, evidence),
        "BM25 (позиція)": first_hit_rank(bm25_search(q, CANDIDATES), gold, evidence),
    })
pd.set_option("display.max_colwidth", 100)
display(pd.DataFrame(rows))

,питання,семантичний (позиція),BM25 (позиція)
0,Що таке самоплагіат?,1.0,2.0
1,Скільки кредитів ЄКТС має бакалаврська програма F2?,9.0,16.0
2,Якому діапазону балів відповідає FX?,NaN,NaN
3,Чи можна поселити котика в кімнату гуртожитку?,4.0,8.0
4,Коли треба заплатити за житло на весняний семестр?,1.0,7.0
5,О котрій годині в гуртожитку потрібно не шуміти?,2.0,14.0


In [18]:
# Питання 27: додаємо до бенчмарку три власні питання
MY_BENCHMARK = [
    ("Хто може бути старостою академічної групи?", ["336"], r"притаманні високі",
     "Студент, якому притаманні високі моральні якості, що користується авторитетом і повагою колег."),
    ("Який крайній термін оплати проживання в гуртожитку за осінній семестр?", ["348"], r"1 вересня",
     "До 1 вересня поточного року."),     # перефразування питання про оплату за перший семестр
    ("Скільки коштує парковка біля університету?", [], None, None),     # відповіді немає
]
BENCHMARK_EXT = BENCHMARK + MY_BENCHMARK
answerable_old = answerable
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK_EXT if gold]
print(f"Питань: {len(BENCHMARK_EXT)}, з відповіддю: {len(answerable)}")

methods = [vector_search, bm25_search, hybrid_search, rerank_search]
before = pd.DataFrame([retrieval_metrics(s) for s in methods]).set_index("метод")
answerable = answerable_old  # для порівняння «до»
before = pd.DataFrame([retrieval_metrics(s) for s in methods]).set_index("метод")
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK_EXT if gold]
after = pd.DataFrame([retrieval_metrics(s) for s in methods]).set_index("метод")
display(pd.concat({"23 питання": before, "25 питань": after}, axis=1).round(2))

# Як система відповідає на нові питання
for question, gold, _, reference in MY_BENCHMARK:
    describe_failure(pd.Series(evaluate(question, gold, reference)))

Питань: 29, з відповіддю: 25


23 питання             25 питань            
                   hit@1 hit@5   MRR     hit@1 hit@5   MRR
метод                                                     
vector_search       0.61  0.87  0.74      0.64  0.88  0.76
bm25_search         0.52  0.65  0.59      0.56  0.68  0.62
hybrid_search       0.61  0.91  0.76      0.64  0.92  0.78
rerank_search       0.83  0.96  0.90      0.84  0.96  0.90

Питання: Хто може бути старостою академічної групи?
  правильний документ: ['336'], знайдено: ['336', '336', '336', '336', '336'] — ретрив знайшов
  правильно: True, цитування: True
  відповідь: Старостою академічної групи може бути студент, якому притаманні високі моральні якості, що користується авторитетом і повагою у своїх колег, має позитивний вплив на колектив групи та володіє організаторськими здібностями [1].

Питання: Який крайній термін оплати проживання в гуртожитку за осінній семестр?
  правильний документ: ['348'], знайдено: ['348', '348', '1219', '348', '348'] — ретрив знайшов
  правильно: True, цитування: True
  відповідь: Крайній термін оплати проживання в гуртожитку за осінній семестр — до 1 вересня поточного року [1].

Питання: Скільки коштує парковка біля університету?
  правильний документ: [], знайдено: ['348', '348', '348', '348', '348'] — ретрив знайшов
  правильно: True, цитування: None
  відповідь: У наданих документах немає відповіді.



### 27. Три власні питання в бенчмарку: як змінилися метрики ретриву?

Додано три питання:
1. **З відповіддю:** «Хто може бути старостою академічної групи?» (документ 336);
2. **Перефразування наявного:** «Який крайній термін оплати проживання в гуртожитку за осінній семестр?» — інакше сформульоване питання «До якої дати потрібно оплатити гуртожиток за перший семестр?» (документ 348);
3. **Без відповіді в документах:** «Скільки коштує парковка біля університету?».

Бенчмарк став: 29 питань, з них 25 з відповіддю (для метрик ретриву враховуються лише вони).

| Метод | hit@1 (23 → 25) | hit@5 (23 → 25) | MRR (23 → 25) |
|---|---|---|---|
| vector_search | 0.61 → 0.64 | 0.87 → 0.88 | 0.74 → 0.76 |
| bm25_search | 0.52 → 0.56 | 0.65 → 0.68 | 0.59 → 0.62 |
| hybrid_search | 0.61 → 0.64 | 0.91 → 0.92 | 0.76 → 0.78 |
| rerank_search | 0.83 → 0.84 | 0.96 → 0.96 | 0.90 → 0.90 |

**Як змінилися метрики:** усі метрики **трохи зросли** (на 1–4 пункти), бо обидва нові питання з відповіддю виявилися «легкими»: правильний фрагмент знаходиться на першому місці, тому частка влучань зросла. Порядок методів не змінився: переранжування найкраще, BM25 найгірший.

**Як відповіла система:**
| Питання | Знайдено | Відповідь | Оцінка |
|---|---|---|---|
| Хто може бути старостою? | 336 ×5 | «Старостою академічної групи може бути студент, якому притаманні високі моральні якості, що користується авторитетом і повагою у своїх колег… [1]» | ✅ правильно, цитування коректне |
| Крайній термін оплати за осінній семестр | 348, 348, 1219, 348, 348 | «…до 1 вересня поточного року [1]» | ✅ правильно. Модель зрозуміла, що «осінній семестр» = «перший семестр», хоча в документі сказано саме «за 1 семестр» |
| Скільки коштує парковка? | 348 ×5 (нерелевантні) | «У наданих документах немає відповіді.» | ✅ правильна відмова |

**Висновок:** перефразування не зламало пошук, бо семантичний пошук і переранжування розуміють синоніми. Питання без відповіді правильно оброблено завдяки правилу відмови. Але три питання змінюють метрики лише на кілька пунктів: одне питання тепер «важить» 4 пункти (1/25). Щоб метрики були надійними, бенчмарк має бути більшим і містити й складні питання, а не лише легкі.

In [19]:
# Питання 29: вердикти судді поруч з еталоном і відповіддю — перевіряємо вручну
check = evaluation.copy()
check["еталон"] = [reference for _, _, _, reference in BENCHMARK]
pd.set_option("display.max_colwidth", 200)
display(check[check["документи"].astype(bool)][["питання", "еталон", "відповідь", "правильно"]].head(8))

,питання,еталон,відповідь,правильно
0,До якої дати потрібно оплатити гуртожиток за перший семестр?,До 1 вересня поточного року.,До 1 вересня поточного року потрібно оплатити гуртожиток за перший семестр [1].,True
1,До якої дати оплачують гуртожиток за другий семестр?,До 1 лютого поточного року.,До 1 лютого поточного року [1].,True
2,З якої до якої години в гуртожитку має бути тиша?,З 23:00 до 7:00.,З 23:00 години до 7:00 години в гуртожитку повинна додержуватись тиша [1].,True
3,У який час гості можуть перебувати в гуртожитку?,З 8:00 до 22:00.,Гості можуть перебувати в гуртожитку з 08:00 до 22:00 години [1].,True
4,Чи можна тримати в гуртожитку домашню тварину?,"Ні, утримувати тварин у гуртожитку заборонено.","Ні, в гуртожитку домашню тварину тримати заборонено [1].",True
5,На який строк укладають договір найму житла в гуртожитку?,На один навчальний рік.,Договір найму житла в гуртожитку укладається терміном на один навчальний рік [1].,True
6,За який час потрібно звільнити гуртожиток після відрахування?,Протягом двох тижнів від дня виходу наказу.,Потрібно звільнити гуртожиток протягом двох тижнів від дня виходу відповідно наказу [1].,True
7,Коли обирають старосту на першому курсі?,"На початку навчального року, після двох тижнів навчального процесу.",Обирають старосту на першому курсі на початку навчального року після двох тижнів навчального процесу [1].,True


### 29. Суддя — та сама модель, що генерує відповіді. Які проблеми? Ручна перевірка вердиктів

**Проблеми, коли суддя — та сама модель:**
1. **Однакові «сліпі плями».** Якщо модель неправильно зрозуміла фрагмент чи термін під час генерації, вона, найімовірніше, так само помилиться й під час оцінювання і визнає свою помилку правильною відповіддю.
2. **Упередженість на користь власного стилю** (self-preference bias): моделі схильні вище оцінювати тексти, схожі на ті, що генерують самі.
3. **Нестабільність і чутливість до формулювань:** вердикт залежить від промпту судді. Відповідь з додатковими деталями або іншими словами суддя може визнати неправильною, як сталося з питанням про стипендію ректора (питання 30).
4. **Мала модель (4B) як суддя** гірше розуміє тонкі відмінності: «не пізніше 1 вересня» і «до 1 вересня», зайві деталі проти суперечності.
5. **Обмеження формату:** суддя відповідає «так» / «ні», без пояснення, тож неможливо зрозуміти, чому він так вирішив. Код бере лише початок відповіді (`startswith("так")`), і будь-яка інша форма («Так.», «відповідь: так») може бути прочитана неправильно.

Тому вердикти судді треба вибірково перевіряти вручну, а краще використовувати суддею **іншу, сильнішу модель**.

**Ручна перевірка.** Порівнюю відповідь з еталоном сама і зіставляю з вердиктом судді:

| # | Питання | Еталон | Відповідь системи | Суддя | Мій вердикт |
|---|---|---|---|---|---|
| 0 | До якої дати оплатити гуртожиток за перший семестр? | До 1 вересня поточного року. | До 1 вересня поточного року… [1] | ✅ так | ✅ правильно |
| 2 | З якої до якої години в гуртожитку тиша? | З 23:00 до 7:00. | З 23:00 години до 7:00 години… [1] | ✅ так | ✅ правильно |
| 3 | У який час гості можуть перебувати в гуртожитку? | З 8:00 до 22:00. | …з 08:00 до 22:00 години [1] | ✅ так | ✅ правильно («08:00» = «8:00») |
| 6 | За який час звільнити гуртожиток після відрахування? | Протягом двох тижнів від дня виходу наказу. | …протягом двох тижнів від дня виходу відповідного наказу [1] | ✅ так | ✅ правильно |
| 7 | Коли обирають старосту на першому курсі? | На початку навчального року, після двох тижнів навчального процесу. | …на початку навчального року після двох тижнів навчального процесу [1] | ✅ так | ✅ правильно |
| — | Для кого встановлюється стипендія ректора? | Для студентів денної форми навчання із суттєвими досягненнями… | Те саме + «є переможцями міжнародних та всеукраїнських олімпіад…» [1] | ❌ ні | ✅ **правильно**: містить усе з еталона плюс деталі з документа |

**Результат:** у 5 з 6 перевірених випадків мій вердикт збігся із суддею. У простих питаннях з однозначною відповіддю (дата, години) суддя надійний. Розбіжність — в одному питанні, де відповідь **довша й детальніша за еталон**: суддя помилково вважав додаткові деталі невідповідністю. Тобто суддя схильний до **хибних «ні»** на розгорнутих відповідях. Метрика «91% правильних» дещо **занижена**, і реальна точність системи вища (22 з 23 ≈ 96%).

In [20]:
# Питання 24 (друга спроба): питання з рідкісними точними словами, де BM25 може бути кращим
more_questions = [
    ("Якому діапазону балів відповідає FX?", ["275"], r"35\s*[-–—]\s*59"),
    ("Що таке фабрикація?", ["1199"], r"[Фф]абрикаці"),
    ("Що таке фальсифікація?", ["1199"], r"[Фф]альсифікаці"),
    ("Чи потрібна довідка про флюорографію для поселення в гуртожиток?", ["348"], r"флюорограф"),
    ("Чи потрібна довідка форми 063 для поселення?", ["348"], r"063"),
    ("Що таке академічна мобільність?", ["340", "277", "275"], r"академічн\w* мобільн"),
    ("Яку допомогу отримує студент в академвідпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної"),
]
rows = []
for q, gold, evidence in my_questions + more_questions:
    rows.append({
        "питання": q,
        "семантичний (позиція)": first_hit_rank(vector_search(q, CANDIDATES), gold, evidence),
        "BM25 (позиція)": first_hit_rank(bm25_search(q, CANDIDATES), gold, evidence),
    })
display(pd.DataFrame(rows))

,питання,семантичний (позиція),BM25 (позиція)
0,Що таке самоплагіат?,1.0,2.0
1,Скільки кредитів ЄКТС має бакалаврська програма F2?,9.0,16.0
2,Якому діапазону балів відповідає FX?,NaN,NaN
3,Чи можна поселити котика в кімнату гуртожитку?,4.0,8.0
4,Коли треба заплатити за житло на весняний семестр?,1.0,7.0
5,О котрій годині в гуртожитку потрібно не шуміти?,2.0,14.0
6,Якому діапазону балів відповідає FX?,NaN,2.0
7,Що таке фабрикація?,1.0,2.0
8,Що таке фальсифікація?,1.0,2.0
9,Чи потрібна довідка про флюорографію для поселення в гуртожиток?,2.0,2.0


### 24. Питання, на якому BM25 знаходить правильний фрагмент вище за семантичний пошук, і навпаки

Позиція першого правильного фрагмента серед 20 кандидатів (менше — краще; «—» — не знайдено серед 20):

| Питання | Семантичний | BM25 | Хто кращий |
|---|---|---|---|
| Чи потрібна довідка **форми 063** для поселення? | 2 | **1** | **BM25** |
| Якому діапазону балів відповідає **FX**? | — | **2** | **BM25** |
| Яку допомогу отримує студент в академвідпустці **за медичними показаннями**? | 5 | **2** | **BM25** |
| Коли треба заплатити за житло на **весняний семестр**? | **1** | 7 | **семантичний** |
| О котрій годині в гуртожитку потрібно **не шуміти**? | **2** | 14 | **семантичний** |
| Чи можна поселити **котика** в кімнату гуртожитку? | **4** | 8 | **семантичний** |
| Що таке самоплагіат? / фабрикація? / фальсифікація? | **1** | 2 | семантичний (трохи) |
| Скільки кредитів ЄКТС має бакалаврська програма F2? | **9** | 16 | семантичний |

**Чому BM25 кращий у перших трьох питаннях:** у них є **рідкісні точні слова, коди й назви**: «063» (номер форми довідки), «FX» (оцінка ЄКТС), «медичними показаннями» (стала юридична фраза). BM25 рахує **точні збіги слів** і дає велику вагу словам, які **рідко** трапляються в корпусі. «063» чи «FX» є лише в кількох фрагментах, тож фрагмент з таким словом одразу отримує високу оцінку. Модель ембедингів кодує **загальний зміст**: «063» чи «FX» для неї лише невеликі деталі, які мало впливають на вектор, і фрагменти на ту саму тему (поселення, оцінювання) без цього коду можуть опинитися вище. Для «FX» семантичний пошук узагалі не знайшов правильний фрагмент серед 20.

**Чому семантичний пошук кращий у питаннях про весняний семестр, тишу й котика:** вони сформульовані **іншими словами, ніж у документах**: «весняний семестр» замість «2 семестр», «не шуміти» замість «дотримуватись тиші», «поселити котика» замість «утримувати тварин». Дослівних збігів мало, і BM25 їх не знаходить (позиції 7, 14, 8). Ембединги ж розуміють **синоніми й зміст**: «котик» → «тварина», «не шуміти» → «тиша». BM25 заважає і те, що українські слова мають багато форм («семестр» / «семестрі» / «семестру» для нього різні слова).

**Висновок:** методи помиляються на **різних** питаннях, тому їх поєднують у **гібридний пошук** (RRF, розділ 5). BM25 «рятує» питання з кодами й термінами, а семантичний пошук — питання, сформульовані своїми словами. Саме тому в розділі 7 гібридний пошук має кращий hit@5 (0.91), ніж кожен метод окремо.